# 02: STRING Interaction Fetching

This notebook handles:
- Loading high pLLPS proteins from previous step
- Fetching protein-protein interactions from STRING database
- Matching interactions to pLLPS dataset
- Saving interaction data for downstream analysis

**Inputs:**
- `results/high_pllps_proteins.csv` - From notebook 01
- `results/full_dataset.csv` - From notebook 01

**Outputs:**
- `results/string_interactions_raw.csv` - Raw STRING interactions
- `results/string_interactions_matched.csv` - Interactions matched to pLLPS data
- `results/interaction_summary.json` - Summary statistics

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import llps_functions as lf
from pathlib import Path

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Previous Results

In [ ]:
# Check available results
lf.list_saved_results()

In [ ]:
# Load datasets
df_full = lf.load_analysis_result('full_dataset', format='csv')
df_high = lf.load_analysis_result('high_pllps_proteins', format='csv')

print(f"\n📊 Loaded data:")
print(f"   Full dataset: {len(df_full)} proteins")
print(f"   High pLLPS: {len(df_high)} proteins")

## 2. Fetch STRING Interactions

⚠️ **Note:** This step makes API calls to STRING database. It may take several minutes depending on the number of proteins. Results will be cached to avoid re-fetching.

In [ ]:
# Configuration
SCORE_THRESHOLD = 700  # STRING confidence score (400=low, 700=high, 900=highest)
MAX_PROTEINS = 500  # Limit number of proteins to avoid excessive API calls

# Get protein IDs
protein_ids = df_high['Entry'].head(MAX_PROTEINS).tolist()
print(f"\n🔍 Fetching interactions for {len(protein_ids)} high pLLPS proteins...")
print(f"   Score threshold: {SCORE_THRESHOLD}")

In [ ]:
# Check if we already have cached interactions
cache_file = Path('results/string_interactions_raw.csv')

if cache_file.exists():
    print("\n✅ Found cached interactions, loading from file...")
    interactions_df = lf.load_analysis_result('string_interactions_raw', format='csv')
    errors = []
else:
    print("\n⏳ Fetching interactions from STRING API...")
    print("   This may take several minutes. Progress will be shown below.")
    
    # Fetch interactions with progress callback
    interactions_df, errors = lf.fetch_string_interactions(
        protein_ids,
        score_threshold=SCORE_THRESHOLD,
        progress_callback=lambda msg: print(f"   {msg}")
    )
    
    if len(interactions_df) > 0:
        # Save raw interactions
        lf.save_analysis_result(interactions_df, 'string_interactions_raw', format='csv')
    else:
        print("\n⚠️  No interactions found!")

In [ ]:
# Show fetch results
print(f"\n📊 STRING Interaction Results:")
print(f"   Total interactions: {len(interactions_df)}")
print(f"   Unique proteins with interactions: {len(set(interactions_df['protein1'].tolist() + interactions_df['protein2'].tolist()))}")

if errors:
    print(f"   ⚠️  Errors encountered: {len(errors)}")
    print(f"   First few errors: {errors[:3]}")

# Display sample interactions
print("\n📋 Sample interactions:")
interactions_df.head(10)

In [ ]:
# Visualize score distribution
if len(interactions_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Score histogram
    axes[0].hist(interactions_df['score'], bins=50, edgecolor='black', alpha=0.7)
    axes[0].axvline(SCORE_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({SCORE_THRESHOLD})')
    axes[0].set_xlabel('STRING Confidence Score')
    axes[0].set_ylabel('Count')
    axes[0].set_title('STRING Score Distribution')
    axes[0].legend()
    
    # Interactions per protein
    protein_counts = pd.concat([
        interactions_df['protein1'].value_counts(),
        interactions_df['protein2'].value_counts()
    ]).groupby(level=0).sum().sort_values(ascending=False)
    
    axes[1].hist(protein_counts.values, bins=30, edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Number of Interactions')
    axes[1].set_ylabel('Number of Proteins')
    axes[1].set_title('Interactions per Protein')
    
    plt.tight_layout()
    plt.savefig('results/string_interaction_stats.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✅ Saved plot: results/string_interaction_stats.png")

## 3. Match Interactions to pLLPS Dataset

In [ ]:
# Match interactions to full dataset
if len(interactions_df) > 0:
    print("\n🔗 Matching interactions to pLLPS dataset...")
    
    matched_df = lf.match_interactions_to_pllps(interactions_df, df_full)
    
    print(f"\n✅ Matching complete:")
    print(f"   Total interactions: {len(matched_df)}")
    print(f"   Both partners in dataset: {matched_df['both_in_dataset'].sum()}")
    print(f"   Interactions with pLLPS scores: {matched_df[['pllps_1', 'pllps_2']].notna().all(axis=1).sum()}")
else:
    matched_df = pd.DataFrame()
    print("\n⚠️  No interactions to match")

In [ ]:
# Show matched data
if len(matched_df) > 0:
    print("\n📋 Sample matched interactions:")
    display(matched_df[['protein1', 'protein2', 'score', 'pllps_1', 'pllps_2', 'both_in_dataset']].head(10))
    
    # Save matched interactions
    lf.save_analysis_result(matched_df, 'string_interactions_matched', format='csv')

In [ ]:
# Visualize pLLPS scores of interacting proteins
if len(matched_df) > 0:
    complete_pairs = matched_df[matched_df['both_in_dataset']].copy()
    
    if len(complete_pairs) > 0:
        fig, ax = plt.subplots(figsize=(10, 8))
        
        scatter = ax.scatter(
            complete_pairs['pllps_1'], 
            complete_pairs['pllps_2'],
            c=complete_pairs['score'],
            cmap='viridis',
            alpha=0.6,
            s=50
        )
        
        # Add threshold lines
        ax.axhline(0.7, color='red', linestyle='--', alpha=0.5, label='High pLLPS threshold')
        ax.axvline(0.7, color='red', linestyle='--', alpha=0.5)
        
        ax.set_xlabel('pLLPS Score (Protein 1)')
        ax.set_ylabel('pLLPS Score (Protein 2)')
        ax.set_title('pLLPS Scores of Interacting Protein Pairs')
        ax.legend()
        
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('STRING Confidence Score')
        
        plt.tight_layout()
        plt.savefig('results/pllps_interaction_scatter.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        print("\n✅ Saved plot: results/pllps_interaction_scatter.png")

## 4. Save Summary Statistics

In [ ]:
# Create summary
if len(matched_df) > 0:
    complete_pairs = matched_df[matched_df['both_in_dataset']]
    
    summary = {
        'proteins_queried': len(protein_ids),
        'total_interactions': len(interactions_df),
        'matched_interactions': len(matched_df),
        'complete_pairs': len(complete_pairs),
        'score_threshold': SCORE_THRESHOLD,
        'unique_proteins_with_interactions': len(set(interactions_df['protein1'].tolist() + interactions_df['protein2'].tolist())),
        'errors_count': len(errors),
    }
    
    lf.save_analysis_result(summary, 'interaction_summary', format='json')
    
    print("\n" + "="*60)
    print("✅ All results saved successfully!")
    print("="*60)
    print(f"\n📊 Summary:")
    for key, value in summary.items():
        print(f"   {key}: {value}")
else:
    print("\n⚠️  No interactions to summarize")

In [ ]:
# List all saved files
lf.list_saved_results()

## Summary

✅ **Completed:**
1. Loaded high pLLPS proteins from previous analysis
2. Fetched protein-protein interactions from STRING database
3. Matched interactions to pLLPS dataset
4. Generated visualizations
5. Saved all results to `results/` directory

**Next step:** Run `03_enrichment_analysis.ipynb` to test for high-high pLLPS interaction enrichment.